# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [23]:
# constants

# MODEL = 'openai/gpt-5-nano'
# MODEL = 'google/gemma-4-31b-it:free'
# MODEL = 'openai/gpt-oss-120b'
MODEL = 'z-ai/glm-5'

In [24]:
# set up environment
# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')
if api_key and api_key.startswith('sk-or-') and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

OLLAMA_MODEL = "gemma4:latest"

API key looks good so far


In [25]:
# here is the question; type over this to ask something new

term = "one-shot prompting"
question = f"""
Define "{term}" in plain language, then:
1. Give a real-world example of it in use.
2. Relate it to cooking with an analogy that maps the core mechanism accurately — not just a surface-level comparison.
"""

In [26]:
# Get gpt-4o-mini to answer, with streaming

stream = openai.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
    stream=True
)

reply = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    reply += chunk.choices[0].delta.content or ''
    reply = reply.replace("```", "").replace("markdown", "")
    update_display(Markdown(reply), display_id=display_handle.display_id)

**Definition**

**One-shot prompting** is a technique where you give an AI model **exactly one specific example** of the task you want it to perform before asking it to do the task itself.

Instead of just giving instructions (which is "zero-shot"), you provide a single "solved" problem to show the AI exactly what you mean. This helps the AI understand the pattern, style, or format you expect in its answer.

***

### 1. Real-World Example

Imagine you want an AI to classify customer reviews as either "Positive" or "Negative."

**Without One-Shot (Zero-Shot):**
You ask: *"Tell me if this review is positive or negative: 'The food was cold but the service was fast.'"*
*   **Risk:** The AI might just say "It’s mixed," or "It leans negative," because it doesn't know you want a strict binary label.

**With One-Shot Prompting:**
You provide this prompt:
> **Example:** Review: "I loved the atmosphere, but the waiter was rude." Sentiment: Negative.
> **Task:** Review: "The food was cold but the service was fast." Sentiment:

**The Result:** The AI looks at your single example. It sees that you provided a short, decisive answer ("Negative") rather than a long explanation. It then applies that same format to your new request.
*   **AI Response:** "Positive."

By showing the AI just one example, you taught it two things: the definition of the sentiment *and* the specific output format you wanted.

***

### 2. The Cooking Analogy

To understand the **core mechanism** of one-shot prompting, compare it to teaching a new line cook how to plate a dish.

**The Setup:**
Imagine a skilled chef (the AI) who knows how to cook thousands of recipes. However, every restaurant has different rules for how the food should look when it leaves the kitchen.

**Zero-Shot (No Example):**
The Head Chef (You) yells, *"Plate the risotto!"*
*   **What happens:** The cook uses their general knowledge. They might put the risotto in a bowl, or they might spread it flat on a plate. They do the job, but they have to guess the specific style the restaurant prefers. They might get it wrong.

**One-Shot Prompting (The Core Mechanism):**
The Head Chef (You) walks over to a plate that is already finished and sitting on the counter. They point to it and say, *"Plate the risotto exactly like **this one**."*
*   **The Mechanism:** The cook (AI) stops guessing. They examine that **single reference plate** (the "one-shot"). They notice the sauce is drizzled in a circle and the parsley is chopped finely, not whole.
*   **The Result:** The cook uses their general skill to cook the risotto, but they apply the **specific pattern** they learned from looking at that one example to the new dish.

**Why this maps accurately:**
It isn't just about showing the cook a picture; it's about constraining their options. The AI (cook) already has the "recipe" (data) in its head, but the one-shot example acts as a **template** or a mold. It forces the AI to conform to a specific structure or style immediately, without needing a whole cookbook of examples (few-shot) to figure out the pattern.

In [19]:
# Get Gemma to answer, with streaming

stream = ollama.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": question}],
    stream=True
)

reply = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    reply += chunk.choices[0].delta.content or ''
    reply = reply.replace("```", "").replace("markdown", "")
    update_display(Markdown(reply), display_id=display_handle.display_id)

## Defining One-Shot Prompting

**Definition in Plain Language:**
One-shot prompting is a technique used when talking to large language models (LLMs) like advanced chatbots, where instead of just telling the model what you want it to do ("Instructions"), you proactively give it **one perfect example** of the desired input and output. It’s basically showing the model exactly how the work should look before asking it to perform the task on its own data.

The core idea is that by providing one well-structured sample, the model doesn't have to guess your format, tone, or logic—it learns the pattern from observing the single example and then applies that learned pattern to all subsequent requests.

***

### 1. Real-World Example in Use (Data Extraction)

Imagine you are working with thousands of customer service emails and need an AI model to extract specific data points (like names, dates, and product IDs) into a clean table format.

**The Challenge:** If you simply prompt the model saying, "Extract names and product IDs," it might interpret that loosely or hallucinate things.
**One-Shot Solution:** You provide one meticulously formatted example first:

> **[Input Example]**
> *“I received my order today, October 15th, for the new Model Z coffee maker (ID: X47).”*
>
> **[Output Specification]**
> *Name: N/A | Date Acquired: 2023-10-15 | Product ID: X47*

Now, when you feed it a second email, the model is much more likely to follow that exact structure (e.g., populating `Name: [Found Name]`, keeping the date format consistent, and correctly isolating the product ID) because it has been provided with a template of perfect execution.

***

### 2. The Cooking Analogy

The core function of one-shot prompting is **teaching by demonstration.** It's not just about comparing ingredients; it’s about controlling the *process* or the *final structure*.

**Analogy: Learning to Plate Food (Plating Theory)**

Imagine you are a mentor teaching an apprentice how to plate a complex, gourmet dish for presentation.

*   **The Task:** The apprentice must arrange the side components—some roasted vegetables and some delicate sauce—on a bespoke porcelain plate so that the chef review deems it elegant and balanced.
*   **The Problem (Zero-Shot):** If you only say, "Arrange these vegetables and this reduction sauce," the apprentice could scatter them haphazardly or use too much space, resulting in an unappetizing mess. The model output would be unpredictable.
*   **One-Shot Prompting:** Instead of just giving verbal instructions, you give the apprentice a **perfect reference plate.** You say: "Here is how we do it in this restaurant. See? The sauce forms a *swirl starting from the 7 o'clock position*, and the roasted carrots are placed *parallel to the edge*." (This single example dictates the desired spatial relationship.)

**The Mapping:**
1. **Instruction Set = General Concept:** You want an appealing plate.
2. **Reference Plate (One-Shot Example) = The Prompt Sample:** This sample doesn't just show the ingredients; it shows the *system of arrangement*—the exact pattern, shape, flow, and spacing that must be maintained.
3. **The Apprentice’s Work (New Input) = Unseen Data:** When a new set of vegetables comes in, the apprentice doesn't simply dump them on the plate. They replicate the learned rule from the sample: maintain the perfect negative space, keep the swirl at 7 o'clock, and arrange things parallel to the edge.

In both cases (prompting and plating), you are not just providing ingredients or commands; **you are supplying a blueprint of successful execution.**